In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from google import genai
client=genai.Client()
MODEL="gemini-3.1-flash-lite"

In [3]:
from typing import Literal
from pydantic import BaseModel,Field

In [4]:
from dataclasses import field
class Triage(BaseModel):
    category: Literal["billing","technical","account","general"]= Field(description="The main topic of the customer's message.")
    urgency: Literal["low","mediam","high"]=Field(description="How quickly this needs a response")
    sentiment: Literal["negative","neutral","positive"]=Field(description="The customers mood in the message")
    need_human:bool=Field(description="True if this must be escalated to a human — refunds, billing disputes, "
                    "cancellations, legal/privacy issues, or an angry customer.")
    summary:str=Field(description="one line summary what the customer wants")

In [5]:
import yaml
from pathlib import Path
_PROMPTS = yaml.safe_load((Path.cwd() / "prompt.yml").read_text(encoding="utf-8"))
TRIAGE_SYSTEM = _PROMPTS["triage_system"]
REPLY_SYSTEM = _PROMPTS["reply_system"] 

print("Prompts loaded successfully!")


Prompts loaded successfully!


In [6]:
IN_PRICE_PER_MTOK = 0.25     
OUT_PRICE_PER_MTOK = 1.50    
_totals = {"input": 0, "output": 0}
def track(usage) -> None:
    _totals["input"] += usage.prompt_token_count or 0
    _totals["output"] += usage.candidates_token_count or 0
def usage_report() -> str:
    cost = (_totals["input"] / 1e6 * IN_PRICE_PER_MTOK
            + _totals["output"] / 1e6 * OUT_PRICE_PER_MTOK)
    return f"{_totals['input']} in + {_totals['output']} out tokens = ~${cost:.10f}"

In [7]:
def triage(message: str) -> Triage:
    resp = client.models.generate_content(
        model=MODEL,
        contents=message,
        config={
            "system_instruction": TRIAGE_SYSTEM,
            "response_mime_type": "application/json",  
            "response_schema": Triage,                 
        },
    )
    track(resp.usage_metadata)      
    return resp.parsed

In [8]:
def draft_reply(message: str, t: Triage) -> str:
    resp = client.models.generate_content(
        model=MODEL,
        contents=f"Customer message:\n{message}\n\nTriage: {t.model_dump()}",
        config={"system_instruction": REPLY_SYSTEM},
    )
    track(resp.usage_metadata)      
    return resp.text

In [11]:
def main() -> None:
    samples = [
        "I've been charged twice for May and nobody has replied. This is ridiculous.",
        "How do I change the email address on my account?",
        "Your app keeps crashing when I upload a file.",
        "Who are you?",
        "What can you do for me",
        "Can I directly contact with the human support",
    ]
    for s in samples:
        t = triage(s)
        print("-" * 70)
        print("MESSAGE :", s)
        print("TRIAGE  :", t.model_dump())
        print("ESCALATE:", "yes -> human" if t.need_human else "no")
        print("DRAFT   :", draft_reply(s, t))

    print("=" * 70)
    print("TOKENS  :", usage_report())

main()

----------------------------------------------------------------------
MESSAGE : I've been charged twice for May and nobody has replied. This is ridiculous.
TRIAGE  : {'category': 'billing', 'urgency': 'high', 'sentiment': 'negative', 'need_human': True, 'summary': 'Customer reports a double charge for May and expresses frustration over lack of previous response.'}
ESCALATE: yes -> human
DRAFT   : Hello, I am very sorry for the frustration caused by this billing error and the delay in our response. A specialized teammate is currently reviewing your account details to resolve this matter for you, and they will follow up shortly with the next steps.
----------------------------------------------------------------------
MESSAGE : How do I change the email address on my account?
TRIAGE  : {'category': 'account', 'urgency': 'low', 'sentiment': 'neutral', 'need_human': False, 'summary': 'Customer is asking for instructions on how to update their account email address.'}
ESCALATE: no
DRAFT   